In [ ]:
import re
from archived.declaration import VariableDeclaration, HybridProgramDeclaration, PredicateDeclaration
from KyxParser import KyxParser

In [ ]:
kyx_path = 'CoastingCar_with_pred.kyx'
kyx_folder_path = 'kyxs'
template = 'env_template.py'

In [3]:
def read_file_as_string(filepath):
    """Reads the entire file into a single string.

    Args:
        filepath: The path to the text file.

    Returns:
        The file content as a string, or None if the file doesn't exist.
    """
    try:
        with open(filepath, 'r') as file:
            file_content = file.read()
            return file_content
    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
        return None

# Example usage:
kyx_content = read_file_as_string(kyx_path)
template_content = read_file_as_string(template)

In [4]:
print(kyx_content)

ArchiveEntry "Coasting Car"
Description "Coasting cars can stop".

Definitions       /* function symbols cannot change their value */
  Real b;         /* braking capabilities */
  Real ep;        /* reaction time */

  /* precondition */
  Bool precondition(Real v, Real m, Real x) <-> speedConstraint(v, m, x) &  v >= 0 & b > 0;
  
  Bool speedConstraint(Real v, Real m, Real x) <-> v^2<=2*b*(m-x);

  /* controller */
  HP controller ::= {a:=-b; ++ ?v^2<=2*b*(m-x-ep*v); a:=0;};

  /* step */
  HP stepFunction ::= {x'=v, v'=a, t'=1 & v>=0 & t<=ep};

  /* done */
  Bool doneCondition(Real x, Real m) <-> x <= m;

End.

ProgramVariables  /* program variables may change their value over time */
  Real x;         /* position of the car */
  Real v;         /* velocity of the car */
  Real a;         /* acceleration of the car */
  Real m;         /* stop sign position */
  Real t;         /* clock variable */
End.

Problem           /* differential dynamic logic formula */
    precondition(v,

In [5]:
# Example usage:
parser = KyxParser()
parsed_data = parser.parse(kyx_content) # kyx_content is your KYX file content
print(parsed_data)

Error: Invalid step function format.
res:  {'Definitions': [<declaration.VariableDeclaration object at 0x0000017820080EC0>, <declaration.VariableDeclaration object at 0x000001781FE8F750>, <declaration.PredicateDeclaration object at 0x0000017820081010>, <declaration.PredicateDeclaration object at 0x000001781FE8FED0>, <declaration.HybridProgramDeclaration object at 0x0000017820081160>, <declaration.HybridProgramDeclaration object at 0x00000178200AC190>, <declaration.PredicateDeclaration object at 0x00000178200AC2D0>], 'ProgramVariables': [<declaration.VariableDeclaration object at 0x00000178200AC410>, <declaration.VariableDeclaration object at 0x0000017820029350>, <declaration.VariableDeclaration object at 0x00000178200290F0>, <declaration.VariableDeclaration object at 0x000001782006E0F0>, <declaration.VariableDeclaration object at 0x0000017820041D00>], 'Problem': {'precondition': 'precondition(v, m, x)', 'model': ['{\n        controller', 't := 0', 'stepFunction', '}'], 'invariant': 'sp

In [6]:
import re

def parse_hybrid_system_step(step_string):
    """Parses a hybrid system step specification string into a Python function.

    Args:
        step_string: The input string (e.g., "{x'=v, v'=a, t'=1 & v>=0 & t<=ep}").

    Returns:
        A Python function representing the step, or None if parsing fails.  Prints an error message if parsing fails.
    """
    try:
        # Regular expression to extract equations and guards
        match = re.match(r'\{(.*?)\s*\&\s*(.*?)\}', step_string)
        if not match:
            print("Error: Invalid step function format.")
            return None

        equations_str, guards_str = match.groups()
        equations = equations_str.split(',')
        guards = guards_str.split('&')

        # Extract variable assignments from equations
        assignments = {}
        for eq in equations:
            var, expr = eq.strip().split('=')
            assignments[var.strip().replace("'", "")] = expr.strip()


        # Generate Python code
        python_code = """
def stepFunction(self):
    # Assignment
"""
        for var, expr in assignments.items():
            python_code += f"    self.{var} += self.{expr}\n"

        python_code += """
    # Restriction
"""
        for guard in guards:
            guard = guard.strip()
            var, op, val = re.match(r'(\w+)(>=|<=|>|<|=)(.*)', guard).groups()
            python_code += f"""
    if self.{var} {op} {val}:
        self.{var} = {val}
"""

        # Execute the generated Python code
        print(python_code)
        return python_code

    except (AttributeError, ValueError, re.error) as e:
        print(f"Error parsing step function: {e}")
        return None


In [7]:
# Example usage
step_string = "{x'=v, v'=a, t'=1 & v>=0 & t<=ep}"
step_function = parse_hybrid_system_step(step_string)



def stepFunction(self):
    # Assignment
    self.x += self.v
    self.v += self.a
    self.t += self.1

    # Restriction

    if self.v >= 0:
        self.v = 0

    if self.t <= ep:
        self.t = ep



In [8]:
import re

def _parse_declarations(self, declarations_string):
    """Parses declaration strings (Definitions, ProgramVariables) into a dictionary.
       Handles more complex formats including comments and function definitions.
    """
    declarations = {}
    lines = declarations_string.splitlines()
    in_definition_block = False  # Flag to track if we're inside a definition block

    for line in lines:
        line = line.strip()
        if line.startswith("Definitions"):
            in_definition_block = True
            continue
        elif line == "End.":
            in_definition_block = False
            continue
        elif not line or line.startswith("/*"):  # Ignore empty lines and comments
            continue

        if in_definition_block:
            # Use regex to find declarations within the definition block
            match = re.match(r'\s*(\w+)\s+(\w+);', line) # Matches "type name;"
            if match:
                var_type, var_name = match.groups()
                declarations[var_name] = var_type
            else:
                # Handle cases where the declaration format is not "type name;"
                print(f"Warning: Could not parse declaration: {line}")

    return declarations

# Example Usage:
declarations_string = """
Definitions       /* function symbols cannot change their value */
  Real b;         /* braking capabilities */
  Real ep;        /* reaction time */

  /* precondition */
  Bool precondition(Real v, Real m, Real x) <-> speedConstraint(v, m, x) &  v >= 0 & b > 0;

  Bool speedConstraint(Real v, Real m, Real x) <-> v^2<=2*b*(m-x);

  /* controller */
  HP controller(Real v, Real m, Real x) ::= {a:=-b; ++ ?v^2<=2*b*(m-x-ep*v); a:=0;};

  /* step */
  HP stepFunction(Real v, Real a, Real t) ::= {x'=v, v'=a, t'=1 & v>=0 & t<=ep};

  /* done */
  Bool doneCondition(Real x, Real m) <-> x <= m;

End.
"""

declarations = _parse_declarations(None, declarations_string)  # Assuming 'self' is not needed here.
print(declarations)

{'b': 'Real', 'ep': 'Real'}


In [ ]:
import os

def iterate_files_top_level(directory):
    """Iterates over files in the specified directory only (not recursively)."""
    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        if os.path.isfile(filepath):  # Check if it's a file (not a directory)
            print(f"Processing file: {filepath}")
            # Add your file processing logic here

            content = read_file_as_string(filepath)
            parser = KyxParser()
            parsed_data = parser.parse(content) # kyx_content is your KYX file content
            print(parsed_data)


# Example usage:
iterate_files_top_level(kyx_folder_path)

In [10]:
print(kyx_content)

ArchiveEntry "Coasting Car"
Description "Coasting cars can stop".

Definitions       /* function symbols cannot change their value */
  Real b;         /* braking capabilities */
  Real ep;        /* reaction time */

  /* precondition */
  Bool precondition(Real v, Real m, Real x) <-> speedConstraint(v, m, x) &  v >= 0 & b > 0;
  
  Bool speedConstraint(Real v, Real m, Real x) <-> v^2<=2*b*(m-x);

  /* controller */
  HP controller ::= {a:=-b; ++ ?v^2<=2*b*(m-x-ep*v); a:=0;};

  /* step */
  HP stepFunction ::= {x'=v, v'=a, t'=1 & v>=0 & t<=ep};

  /* done */
  Bool doneCondition(Real x, Real m) <-> x <= m;

End.

ProgramVariables  /* program variables may change their value over time */
  Real x;         /* position of the car */
  Real v;         /* velocity of the car */
  Real a;         /* acceleration of the car */
  Real m;         /* stop sign position */
  Real t;         /* clock variable */
End.

Problem           /* differential dynamic logic formula */
    precondition(v,

In [ ]:
kyx_parser(kyx_content)

NameError: name 'kyx_parser' is not defined

: 